In [1]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
import os

import re
##### GENAI
from dotenv import load_dotenv
import google.generativeai as gen

## LLM for the selfquery
from langchain_google_genai import ChatGoogleGenerativeAI

#### separate eng words without spacing
import wordninja

/usr/lib/python3.12/importlib/__init__.py:90: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  return _bootstrap._gcd_import(name[level:], package, level)
/home/mrosaria/Projects/NLP/SwiftieFriend/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Enviroment / LLM

In [ ]:
def get_google_api_key():
    load_dotenv()
    GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
    if not GOOGLE_API_KEY:
        raise ValueError("Missing GOOGLE_API_KEY in .env") 
    return GOOGLE_API_KEY

api_key = get_google_api_key()
gen.configure(api_key=api_key)


In [ ]:
def get_llm(model_name="gemini-1.5-flash"):

    api_key = get_google_api_key() 

    return ChatGoogleGenerativeAI(
        model=model_name,
        google_api_key=api_key,
        temperature=0.7,
    )


# Load Data

## Cleaning / Preprocessing

In [ ]:
def normalize_lyrics(text):
    text = re.sub(r'\[.*?post-chorus.*?\]', '[Post-Chorus]', text, flags=re.IGNORECASE)
    text = re.sub(r'\[.*?chorus.*?\]', '[Chorus]', text, flags=re.IGNORECASE)
    text = re.sub(r'\[.*?verse.*?\]', '[Verse]', text, flags=re.IGNORECASE)
    text = re.sub(r'\[.*?bridge.*?\]', '[Bridge]', text, flags=re.IGNORECASE)
    text = re.sub(r'\[.*?into.*?\]', '[Intro]', text, flags=re.IGNORECASE)
    text = re.sub(r'\[.*?interlude.*?\]', '[Interlude]', text, flags=re.IGNORECASE)
    text = re.sub(r'\[.*?outro.*?\]', '[Outro]', text, flags=re.IGNORECASE)
    return text

def clean_text(text):
    text = re.sub(r'[\u2000-\u200A\u202F\u205F\u3000]', ' ', text)
    text = normalize_lyrics(text)
    # text  = re.sub(r"\\'", "", text)
    return text

def remove_symbols(s):
    return re.sub(r"[-?_,.]", "", s)

def split_by_capitals(s):
    # Creates keyword list with the words
    s = remove_symbols(s)
    n_cap_letters = len(s)
    keywords = [i.lower() for i in re.findall(r'[A-Z][^A-Z]*', s)]
    if (len(keywords) == 1):
        return keywords
    else:
        return [i.lower() for i in wordninja.split(s)]

def space_song_names(s):
    #Joins the cleaned keywords
    #Convert LavanderHaze -> Lavander Haze , Anti-Hero -> anti hero
    return " ".join(i for i in split_by_capitals(s))



## Loading

In [ ]:
all_albums_path = '..data/Taylor-Swift-Lyrics/data/Albums/'
albums_titles = [i for i in os.listdir(all_albums_path) if  os.path.isdir(os.path.join(all_albums_path, i)) ]

In [ ]:
def load_data_albums(album_path):
    all_album_songs = []
    for song_title in os.listdir(album_path):
        if song_title.endswith('.txt'):
            song_path = os.path.join(album_path, song_title)
            loader = TextLoader(song_path, encoding='utf-8')
            #One document per song
            songs = loader.load()
            album_title = song_path.split('/')[-2]
            
            for song in songs:
                song_name = song_title.split('.txt')[0]
                print(f"Processing song: {song_title} from album: {album_title}")
                if 'Lyrics[' in song.page_content:
                    # Remove initial useless text about contributors and translations
                    song.page_content = '[' + song.page_content.split('Lyrics[')[1] 
                elif '[Prologue]' in song.page_content:
                    song.page_content = song.page_content.split('[Prologue]')[1] 
                   
                #song.page_content ='[' + song.page_content.split('Lyrics[')[1] if (('ContributorsTranslations' in song.page_content) & len(song.page_content.split('Lyrics['))>0) else song.page_content 
                #song.page_content.split('[Verse 1]')[1] if '[Verse 1]' in song.page_content else song.page_content
                #song.page_content = '[Verse 1]' + song.page_content if not song.page_content.startswith('[Verse 1]') else song.page_content
                song.page_content = clean_text(song.page_content)
                song.metadata['album'] = album_title
                song.metadata['album_first_name'] = space_song_names(album_title.split('_')[0])
                if len(album_title.split('_')) > 1:
                    album_second_name = ""
                    for i in range(1, len(album_title.split('_')) ):
                        album_second_name += "".join(space_song_names(album_title.split('_')[i]))
                    song.metadata['album_second_name'] = album_second_name
                else: 
                    song.metadata['album_second_name'] = ""
                song.metadata['album_full_name_splitted'] = space_song_names(album_title)
                song.metadata['album_keywords'] = ", ".join(split_by_capitals(album_title))
                song.metadata['song'] = song_name
                song.metadata['song_keywords'] = ", ".join(split_by_capitals(song_name))
                song.metadata['song_name_spaced'] = space_song_names(song_name)

                #song_title.split('/')[-1].replace('.txt', '')
                all_album_songs.append(song)
    return all_album_songs
#load_data_albums(f"{all_albums_path}/theladieslunchingchapter/")
#a = "THETORTUREDPOETSDEPARTMENT_TheBolterEdition_"


# Split

In [ ]:
############# PROBLEM #########
# The one chuck contains several of the splitters like [Chorus], [Verse], etc. its being based only on the length of the text
def split_text(album_document, chunk_size=500, chunk_overlap=50, return_as_documents=True):
    separators=["[Chorus]", "[Verse]", "[Bridge]", "[Intro]", "[Post-Chorus]", "[Interlude]", "[Outro]" ,"\n\n", "\n", " "]
    

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=separators
    )
    if return_as_documents:
        return text_splitter.split_documents(album_document)
    else:
        return text_splitter.split_text(album_document)
     


# #To Implement
# from transformers import GPT2TokenizerFast

# tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

# def tiktoken_len(text):
#     return len(tokenizer.encode(text))

# splitter = RecursiveCharacterTextSplitter(
#     chunk_size=100,
#     chunk_overlap=20,
#     length_function=tiktoken_len,
#     separators=["[Chorus]", "[Verse]", "\n\n", "\n", " "]

# split_text(load_data_albums(f"{all_albums_path}/theladieslunchingchapter/"))

#  Embedding / Storing

## Loading and Splitting

In [ ]:
chunks = []
for album_path in albums_titles:
    album_path = os.path.join(all_albums_path, album_path)
    chunk = split_text(load_data_albums(f"{album_path}/"))
    chunks.extend(chunk)


Processing song: betty.txt from album: theladieslunchingchapter
Processing song: rightwhereyouleftme.txt from album: theladieslunchingchapter
Processing song: nobody_nocrime.txt from album: theladieslunchingchapter
Processing song: dorothea.txt from album: theladieslunchingchapter
Processing song: august.txt from album: theladieslunchingchapter
Processing song: marjorie.txt from album: theladieslunchingchapter
Processing song: tolerateit.txt from album: Evermore
Processing song: happiness.txt from album: Evermore
Processing song: willow.txt from album: Evermore
Processing song: tisthedamnseason.txt from album: Evermore
Processing song: nobody_nocrime.txt from album: Evermore
Processing song: dorothea.txt from album: Evermore
Processing song: cowboylikeme.txt from album: Evermore
Processing song: goldrush.txt from album: Evermore
Processing song: longstoryshort.txt from album: Evermore
Processing song: coneyisland.txt from album: Evermore
Processing song: champagneproblems.txt from albu

In [ ]:
# texts= [doc.metadata for doc in chunks[:40]]
# texts
    

## Embedding / Vector DB

In [ ]:
model_name = "sentence-transformers/all-mpnet-base-v2"
huggingface_embedding = HuggingFaceEmbeddings(model_name=model_name)

In [ ]:
ids = [str(i) for i in range(0, len(chunks))]

In [ ]:
vectordb = Chroma.from_texts(
    texts= [doc.page_content for doc in chunks],
    embedding=huggingface_embedding,
    metadatas= [doc.metadata for doc in chunks],
    ids=ids,
    persist_directory="../data/chroma_db",  # optional path to persist
    collection_name="taylor_songs_collection"
)    

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
